In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName('Data Aggregation')\
    .getOrCreate()

In [2]:
listings = spark.read.csv(
    "data/listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

In [3]:
listings.groupBy(listings.property_type)\
    .count().orderBy('count', ascending=[False]).show(truncate=False)

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Entire rental unit                |41215|
|Private room in rental unit       |14464|
|Private room in home              |11704|
|Entire home                       |9120 |
|Entire condo                      |8250 |
|Private room in condo             |3189 |
|Entire serviced apartment         |1874 |
|Private room in townhouse         |1195 |
|Room in hotel                     |1113 |
|Entire townhouse                  |1058 |
|Private room in bed and breakfast |491  |
|Private room in guesthouse        |377  |
|Entire loft                       |341  |
|Entire guesthouse                 |221  |
|Room in boutique hotel            |217  |
|Entire guest suite                |177  |
|Private room in guest suite       |174  |
|Private room in loft              |153  |
|Private room in serviced apartment|132  |
|Private room                      |103  |
+----------

In [4]:
import pyspark.sql.functions as F

listings.groupBy(listings.property_type)\
    .agg(F.count('property_type').alias('count'))\
    .orderBy('count', ascending=[False]).show(50, truncate=False)

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Entire rental unit                |41215|
|Private room in rental unit       |14464|
|Private room in home              |11704|
|Entire home                       |9120 |
|Entire condo                      |8250 |
|Private room in condo             |3189 |
|Entire serviced apartment         |1874 |
|Private room in townhouse         |1195 |
|Room in hotel                     |1113 |
|Entire townhouse                  |1058 |
|Private room in bed and breakfast |491  |
|Private room in guesthouse        |377  |
|Entire loft                       |341  |
|Entire guesthouse                 |221  |
|Room in boutique hotel            |217  |
|Entire guest suite                |177  |
|Private room in guest suite       |174  |
|Private room in loft              |153  |
|Private room in serviced apartment|132  |
|Private room                      |103  |
|Room in ap

In [5]:
listings.groupBy(listings.property_type)\
    .agg(
    F.count('property_type').alias('count'),
    F.avg('review_scores_location')
).orderBy('count', ascending=[False])\
    .show(truncate=False)

+----------------------------------+-----+---------------------------+
|property_type                     |count|avg(review_scores_location)|
+----------------------------------+-----+---------------------------+
|Entire rental unit                |41215|4.727437046043664          |
|Private room in rental unit       |14464|4.726647667299953          |
|Private room in home              |11704|4.694735943407702          |
|Entire home                       |9120 |4.727462068965489          |
|Entire condo                      |8250 |4.770284788770394          |
|Private room in condo             |3189 |4.778348954578206          |
|Entire serviced apartment         |1874 |4.722610024449879          |
|Private room in townhouse         |1195 |4.766042471042479          |
|Room in hotel                     |1113 |4.608689075630254          |
|Entire townhouse                  |1058 |4.81762931034483           |
|Private room in bed and breakfast |491  |4.724728915662653          |
|Priva

In [6]:
reviews = spark.read.csv(
    "data/reviews.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

reviews.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [7]:
for field in reviews.schema:
    print(field)

StructField('listing_id', LongType(), True)
StructField('id', LongType(), True)
StructField('date', DateType(), True)
StructField('reviewer_id', IntegerType(), True)
StructField('reviewer_name', StringType(), True)
StructField('comments', StringType(), True)


In [11]:
listings_reviews = listings.join(
    reviews, listings.id == reviews.listing_id, how='inner'
)
listings_reviews.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [16]:
listings_reviews.groupBy(listings.id, listings.name)\
    .agg(F.count(reviews.id).alias('num_review'))\
    .orderBy('num_review', ascending=[False])\
    .show(truncate=False)

+--------+--------------------------------------------------+----------+
|id      |name                                              |num_review|
+--------+--------------------------------------------------+----------+
|47408549|Double Room+ Ensuite                              |1902      |
|43120947|Private double room with en suite facilities      |1647      |
|19670926|Locke Studio Apartment at Leman Locke             |1443      |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142      |
|46233904|Superior Studio, avg size 23.5 msq                |1002      |
|2659707 |Large Room + Private Bathroom, E3.                |998       |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951       |
|4748665 |Single bedroom near London Stratford              |933       |
|42081759|Micro Studio at Locke at Broken Wharf             |914       |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909       |
|2025844 |Family Friendly Central London Flat      